# RAG PIPELINE

### Setting up the 2 key LangChain objects: `retriever` and `llm`

#### A sidebar on  `"temperature"`:
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed.

Note 2: if we want creativity, we need to use the System Prompt!

In [1]:
%pip install -U -q langchain-ollama langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI # not free
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
import gradio as gr

/Users/pradeepkumar/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. Load your local Embeddings (The "Searcher")

MODEL = 'BAAI/bge-base-en-v1.5'
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

### Connect to Chroma

In [4]:
# 2. Connect to your Chroma DB

DB_NAME = "vector_db"

embeddings = HuggingFaceEmbeddings(model_name = MODEL)
vectorstore = Chroma(persist_directory = DB_NAME, embedding_function = embeddings)

In [5]:
# 3. Create the retriever
retriever = vectorstore.as_retriever()


# 4. Setup your Free LLM (The "Brain")

# llm = ChatOpenAI(temperature = 0, model_name = MODEL) #not free
#run 'ollama pull llama3' in the terminal
llm = ChatOllama(model = 'llama3', temperature = 0)

### These LangChain objects implement the method `invoke()`

In [8]:
retriever.invoke("Who is Avery?")

[Document(id='05b1bb1f-692b-4411-8478-b288bd5dadcf', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial funding.  \n\n- **2016**: **Meets Expectations**  \n  Growth continued, though challenges arose in operational efficiency that required Avery's attention.  \n\n- **2017**: **Developing**  \n  Market competition intensified, and monthly sales metrics were below targets. Avery implemented new strategies which required a steep learning curve.  \n\n- **2018**: **Exceeds Expect

In [7]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery can refer to several individuals, depending on the context. Here are a few notable ones:\n\n1. Avery Brooks: An American actor and musician, best known for his roles as Hawk in the TV series "Spenser: For Hire" and Captain Benjamin Sisko in the TV series "Star Trek: Deep Space Nine".\n2. Avery Johnson: A former professional basketball player and current college basketball coach, who played in the NBA from 1987 to 1993.\n3. Avery Singer: An American artist known for her abstract paintings that explore themes of technology, nature, and human interaction.\n4. Avery Williamson: An American football linebacker who currently plays for the Tennessee Titans.\n\nIf you\'re referring to someone else named Avery, please provide more context or information about who they are, and I\'ll do my best to help!', additional_kwargs={}, response_metadata={'model': 'llama3', 'created_at': '2026-04-19T10:52:39.942863Z', 'done': True, 'done_reason': 'stop', 'total_duration': 88806305

## Time to put this together!

In [9]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [13]:
# this is RAG

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context = context)
    response = llm.invoke([SystemMessage(content = system_prompt), HumanMessage(content = question)])
    return response.content


In [16]:
answer_question("Who is Avery Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been instrumental in guiding the company since its founding in 2015. With a strong background in insurance technology, Avery has led Insurellm to become a leading provider of innovative insurance products and services.'

In [15]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
